In [1]:
!pip install argilla

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for standardwebhooks: filename=standardwebhooks-1.0.1-py3-none-any.whl size=3619 sha256=4649f68617feef95f74cd685db209ed054c8d88583e7b16cb9a236902b90110f
  Stored in directory: /Users/yuxinliu/Library/Caches/pip/wheels/7f/98/8f/cd1f5e35d5c62ddc3295f6aaa04017867411ce1ecff166388a
Successfully built standardwebhooks
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [argilla]


In [10]:
import argilla as rg

HF_TOKEN = "hf"  # only for private spaces

client = rg.Argilla(
    api_url="https://uesing-owner-argilla.hf.space",
    api_key="owner.apikey",
    headers={"Authorization": f"Bearer {HF_TOKEN}"},  # only for private spaces
)

In [11]:
client.me

User(id=UUID('9a55cd17-0e6c-4b6e-906b-50d092c9b7b1') inserted_at=datetime.datetime(2026, 5, 18, 11, 21, 18, 162073) updated_at=datetime.datetime(2026, 5, 18, 11, 21, 18, 162073) username='owner' role=<Role.owner: 'owner'> first_name='owner' last_name=None password=None)

In [12]:
from datasets import load_dataset

data = load_dataset("SetFit/ag_news", split="train")
data.features

train.jsonl:   0%|          | 0.00/33.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

{'text': Value(dtype='string', id=None),
 'label': Value(dtype='int64', id=None),
 'label_text': Value(dtype='string', id=None)}

In [13]:
settings = rg.Settings(
    fields=[rg.TextField(name="text")],
    questions=[
        rg.LabelQuestion(
            name="label", title="Classify the text:", labels=data.unique("label_text")
        ),
        rg.SpanQuestion(
            name="entities",
            title="Highlight all the entities in the text:",
            labels=["PERSON", "ORG", "LOC", "EVENT"],
            field="text",
        ),
    ],
)

In [14]:
dataset = rg.Dataset(name="ag_news", settings=settings)
dataset.create()

/Users/yuxinliu/anaconda3/lib/python3.11/site-packages/argilla/datasets/_resource.py:264: UserWarning: Workspace not provided. Using default workspace: argilla id: 4a4424a9-50d5-4068-91a2-c367b1c58059
  warnings.warn(f"Workspace not provided. Using default workspace: {workspace.name} id: {workspace.id}")


Dataset(id=UUID('cc1225c1-11ca-4b16-a57b-c8406f543ac2') inserted_at=datetime.datetime(2026, 5, 18, 11, 22, 29, 794783) updated_at=datetime.datetime(2026, 5, 18, 11, 22, 31, 752254) name='ag_news' status='ready' guidelines=None allow_extra_metadata=False distribution=OverlapTaskDistributionModel(strategy='overlap', min_submitted=1) workspace_id=UUID('4a4424a9-50d5-4068-91a2-c367b1c58059') last_activity_at=datetime.datetime(2026, 5, 18, 11, 22, 31, 752254))

In [15]:
dataset.records.log(data, mapping={"label_text": "label"})

/Users/yuxinliu/anaconda3/lib/python3.11/site-packages/argilla/records/_io/_datasets.py:265: UserWarning: Record id column not found in Hugging Face dataset. Using row index and split for record ids.
  warnings.warn(


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

/Users/yuxinliu/anaconda3/lib/python3.11/site-packages/argilla/records/_mapping/_mapper.py:89: UserWarning: Keys ['label_text'] in data are not present in the mapping and will be ignored.
  warnings.warn(f"Keys {unknown_keys} in data are not present in the mapping and will be ignored.")
Sending records...: 469batch [39:59,  5.12s/batch]                      


DatasetRecords(Dataset(id=UUID('cc1225c1-11ca-4b16-a57b-c8406f543ac2') inserted_at=datetime.datetime(2026, 5, 18, 11, 22, 29, 794783) updated_at=datetime.datetime(2026, 5, 18, 11, 22, 31, 752254) name='ag_news' status='ready' guidelines=None allow_extra_metadata=False distribution=OverlapTaskDistributionModel(strategy='overlap', min_submitted=1) workspace_id=UUID('4a4424a9-50d5-4068-91a2-c367b1c58059') last_activity_at=datetime.datetime(2026, 5, 18, 11, 22, 31, 752254)))

In [16]:
status_filter = rg.Query(filter=rg.Filter([("status", "==", "completed")]))
filtered_records = dataset.records(status_filter)

In [21]:
from huggingface_hub import login

login(token="hf")

filtered_records.to_datasets().push_to_hub("Uesing/ag_news_annotated")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format: 0ba [00:00, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/Uesing/ag_news_annotated/commit/ca0bdf1d726770760167df89bec8620238be8956', commit_message='Upload dataset', commit_description='', oid='ca0bdf1d726770760167df89bec8620238be8956', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Uesing/ag_news_annotated', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Uesing/ag_news_annotated'), pr_revision=None, pr_num=None)

In [22]:
dataset = rg.Dataset.from_hub("Uesing/ag_news_annotated")

/Users/yuxinliu/anaconda3/lib/python3.11/site-packages/argilla/datasets/_io/_hub.py:356: UserWarning: Open the following URL in your browser to configure the dataset: https://uesing-owner-argilla.hf.space/new/Uesing%2Fag_news_annotated?subset=None&split=None
  warnings.warn(f"Open the following URL in your browser to configure the dataset: {url}")
